In [1]:
import pandas as pd
import numpy as np
from scipy import stats

df=pd.read_csv('housing.csv')


In [2]:
# Produce a custom "Data Quality Report"
print("--- DATA QUALITY REPORT ---")
print(f"Total Rows: {df.shape[0]} | Total Columns: {df.shape[1]}")
print(f"\nDuplicate Rows Count: {df.duplicated().sum()}")
print("\nNull Values Per Column:")
print(df.isnull().sum())
print("\nData Types:")
print(df.dtypes)

--- DATA QUALITY REPORT ---
Total Rows: 545 | Total Columns: 13

Duplicate Rows Count: 0

Null Values Per Column:
price               0
area                0
bedrooms            0
bathrooms           0
stories             0
mainroad            0
guestroom           0
basement            0
hotwaterheating     0
airconditioning     0
parking             0
prefarea            0
furnishingstatus    0
dtype: int64

Data Types:
price               int64
area                int64
bedrooms            int64
bathrooms           int64
stories             int64
mainroad              str
guestroom             str
basement              str
hotwaterheating       str
airconditioning       str
parking             int64
prefarea              str
furnishingstatus      str
dtype: object


In [4]:
# Create a copy to track 'before' metrics
df_cleaned = df.copy()

# Automatically impute numeric missing values with median
numeric_cols = df_cleaned.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df_cleaned[col].isnull().sum() > 0:
        df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].median())

# Automatically impute object/text missing values with mode (most frequent value)
object_cols = df_cleaned.select_dtypes(include=['object']).columns
for col in object_cols:
    if df_cleaned[col].isnull().sum() > 0:
        df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].mode()[0])

print("Missing data handling completed successfully.")

Missing data handling completed successfully.


C:\Users\91704\AppData\Local\Temp\ipykernel_16148\46422008.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df_cleaned.select_dtypes(include=['object']).columns


In [5]:
initial_count = df_cleaned.shape[0]
df_cleaned = df_cleaned.drop_duplicates()
final_count = df_cleaned.shape[0]

print(f"Removed {initial_count - final_count} duplicate rows.")

Removed 0 duplicate rows.


In [6]:
# Convert all text entries to lowercase to standardize comparisons
for col in df_cleaned.select_dtypes(include=['object']).columns:
    df_cleaned[col] = df_cleaned[col].astype(str).str.strip().str.lower()

print("Categorical fields standardized to lowercase.")

Categorical fields standardized to lowercase.


C:\Users\91704\AppData\Local\Temp\ipykernel_16148\1846615454.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_cleaned.select_dtypes(include=['object']).columns:


In [7]:
# Apply IQR outlier capping on critical numeric columns
for col in ['price', 'area']:
    Q1 = df_cleaned[col].quantile(0.25)
    Q3 = df_cleaned[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Cap values beyond thresholds to reduce statistical skew
    df_cleaned[col] = np.clip(df_cleaned[col], lower_bound, upper_bound)

print("Outlier tracking and capping completed using IQR.")

Outlier tracking and capping completed using IQR.


In [8]:
# Convert main counts to integers and pricing metrics explicitly to float types
df_cleaned['price'] = df_cleaned['price'].astype(float)
df_cleaned['area'] = df_cleaned['area'].astype(float)
df_cleaned['bedrooms'] = df_cleaned['bedrooms'].astype(int)
df_cleaned['bathrooms'] = df_cleaned['bathrooms'].astype(int)
df_cleaned['stories'] = df_cleaned['stories'].astype(int)

print("Data structures cleanly cast into accurate object types.")

Data structures cleanly cast into accurate object types.


In [11]:
print(summary_table.to_string(index=False))

         Data Metric  Before Cleaning  After Cleaning
           Row Count              545             545
Total Missing Values                0               0
    Total Duplicates                0               0


In [12]:
df_cleaned.to_csv('housing_cleaned.csv', index=False)
print("Cleaned dataset saved to 'housing_cleaned.csv'.")

Cleaned dataset saved to 'housing_cleaned.csv'.


###  Justification: Missing Data Handling Strategy
- **Numeric Fields (`price`, `area`):** We utilize **Median Imputation** rather than mean imputation. Median imputation protects our baseline metrics from being skewed by extreme luxury house pricing anomalies.
- **Categorical Fields:** We utilize **Mode Imputation** (most frequent value) to handle any missing string attributes. This ensures structural integrity without introducing synthetic categories.
-

###  Justification: Outlier Detection and Mitigation Strategy
- **Method Chosen:** Interquartile Range (IQR) method with a standard threshold of $1.5 \times \text{IQR}$.
- **Decision (Cap vs. Remove):** We chose to **Cap/Clip** extreme price and area entries at our calculated upper and lower boundary thresholds rather than deleting the rows entirely. This successfully eliminates statistical skewing while preserving our overall volume of 545 historical records for down-stream modeling.
-